# Linear Regression From Scratch (NumPy only)

Hypothesis, MSE cost, gradients and the gradient-descent loop are all written by hand with NumPy.
scikit-learn is used **only** for `train_test_split` and for the final sanity-check against `LinearRegression`.

**Contents**
1. Load & clean data
2. Split & standardize
3. The model: hypothesis, cost, gradient, gradient descent
4. Gradient check (proves the gradient formula is right)
5. Simple (1-feature) regression
6. Multiple regression + learning-rate tuning
7. Test-set evaluation (MSE, R² computed manually)
8. Comparison with scikit-learn
9. The math, in plain words

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn: ONLY for the data split, the dataset, and the final comparison
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression   # used only in section 8

np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4.5)

## 1. Load & clean the data
California Housing: 20,640 districts, 8 numeric features, target = median house value (in $100,000s).

In [ ]:
from sklearn.datasets import fetch_california_housing

try:
    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()
    TARGET = "MedHouseVal"
except Exception as e:                       # offline fallback so the notebook still runs
    from sklearn.datasets import load_diabetes
    print("Could not download California housing, falling back to diabetes:", e)
    data = load_diabetes(as_frame=True)
    df = data.frame.copy()
    TARGET = "target"

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Missing values: report, then impute with the column median (none expected here, but the code is general)
print("Missing values per column:\n", df.isna().sum(), "\n")
df = df.fillna(df.median(numeric_only=True))

# Simple duplicate check
print("Duplicate rows:", df.duplicated().sum())
df.describe().T

## 2. Train/test split and standardization
Standardization uses **training-set** mean and std only (no test-set leakage). Without it, gradient descent crawls because features live on very different scales.

In [ ]:
X_df = df.drop(columns=[TARGET])
feature_names = list(X_df.columns)
X = X_df.to_numpy(dtype=float)
y = df[TARGET].to_numpy(dtype=float)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mu = X_train_raw.mean(axis=0)
sigma = X_train_raw.std(axis=0)

def standardize(A):
    return (A - mu) / sigma

X_train = standardize(X_train_raw)
X_test  = standardize(X_test_raw)

def add_bias(A):
    # prepend a column of ones so theta[0] is the intercept
    return np.c_[np.ones(A.shape[0]), A]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train feature means ~0:", np.round(X_train.mean(axis=0), 6))

## 3. The model

With a design matrix $X \in \mathbb{R}^{m\times(n+1)}$ (first column all ones) and parameters $\theta\in\mathbb{R}^{n+1}$:

- Hypothesis: $h_\theta(X) = X\theta$
- Cost: $J(\theta)=\dfrac{1}{2m}\sum_{i=1}^m\big(h_\theta(x^{(i)})-y^{(i)}\big)^2 = \dfrac{1}{2m}(X\theta-y)^\top(X\theta-y)$
- Gradient: $\nabla_\theta J=\dfrac{1}{m}X^\top(X\theta-y)$
- Update: $\theta \leftarrow \theta-\alpha\,\nabla_\theta J$

In [ ]:
def hypothesis(X, theta):
    # h(X) = X @ theta
    return X @ theta

def compute_cost(X, y, theta):
    # J(theta) = 1/(2m) * sum((X theta - y)^2)
    m = len(y)
    residual = hypothesis(X, theta) - y
    return (residual @ residual) / (2 * m)

def compute_gradient(X, y, theta):
    # dJ/dtheta = 1/m * X^T (X theta - y)
    m = len(y)
    return (X.T @ (hypothesis(X, theta) - y)) / m

def gradient_descent(X, y, alpha=0.1, n_iters=1000, theta_init=None, tol=None):
    # Batch gradient descent. Returns (theta, cost_history).
    theta = np.zeros(X.shape[1]) if theta_init is None else theta_init.copy()
    cost_history = [compute_cost(X, y, theta)]
    for _ in range(n_iters):
        theta = theta - alpha * compute_gradient(X, y, theta)
        cost_history.append(compute_cost(X, y, theta))
        if not np.isfinite(cost_history[-1]):          # diverged
            break
        if tol is not None and abs(cost_history[-2] - cost_history[-1]) < tol:
            break
    return theta, np.array(cost_history)

def predict(X, theta):
    return hypothesis(X, theta)

### Evaluation metrics (manual, no `sklearn.metrics`)
$$\text{MSE}=\frac1m\sum(\hat y-y)^2,\qquad R^2=1-\frac{\sum(y-\hat y)^2}{\sum(y-\bar y)^2}$$

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

## 4. Gradient check
Compare the analytic gradient against a finite-difference estimate. If they match, the derivation is correct.

In [ ]:
def numerical_gradient(X, y, theta, eps=1e-5):
    g = np.zeros_like(theta)
    for j in range(len(theta)):
        e = np.zeros_like(theta); e[j] = eps
        g[j] = (compute_cost(X, y, theta + e) - compute_cost(X, y, theta - e)) / (2 * eps)
    return g

Xb_train = add_bias(X_train)
Xb_test  = add_bias(X_test)

rng = np.random.default_rng(0)
theta_rand = rng.normal(size=Xb_train.shape[1])
g_analytic = compute_gradient(Xb_train, y_train, theta_rand)
g_numeric  = numerical_gradient(Xb_train, y_train, theta_rand)
print("Max abs difference:", np.max(np.abs(g_analytic - g_numeric)))
assert np.allclose(g_analytic, g_numeric, atol=1e-6)
print("Gradient check passed ✅")

## 5. Simple linear regression (one feature)
Using the single feature most correlated with the target.

In [ ]:
corrs = pd.Series([np.corrcoef(X_train[:, j], y_train)[0, 1] for j in range(X_train.shape[1])],
                  index=feature_names)
best_feat = corrs.abs().idxmax()
j = feature_names.index(best_feat)
print(f"Most correlated feature: {best_feat} (r = {corrs[best_feat]:.3f})")

Xs_train = add_bias(X_train[:, [j]])
Xs_test  = add_bias(X_test[:, [j]])

theta_simple, cost_simple = gradient_descent(Xs_train, y_train, alpha=0.1, n_iters=500)

print("theta (intercept, slope):", theta_simple)
print("Test MSE:", mse(y_test, predict(Xs_test, theta_simple)))
print("Test R² :", r2_score_manual(y_test, predict(Xs_test, theta_simple)))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(cost_simple); ax[0].set_title("Simple regression: cost vs iteration")
ax[0].set_xlabel("Iteration"); ax[0].set_ylabel("J(θ)")
ax[1].scatter(X_test[:, j], y_test, s=4, alpha=0.3, label="test data")
xs = np.linspace(X_test[:, j].min(), X_test[:, j].max(), 100)
ax[1].plot(xs, theta_simple[0] + theta_simple[1] * xs, "r", lw=2, label="fitted line")
ax[1].set_xlabel(f"{best_feat} (standardized)"); ax[1].set_ylabel(TARGET); ax[1].legend()
ax[1].set_title("Fitted line on test data")
plt.tight_layout(); plt.show()

## 6. Multiple linear regression (all features)
### Learning-rate tuning
Try several learning rates and look at the cost curves: too small = slow, too large = oscillation or divergence.

In [ ]:
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.5]
fig, ax = plt.subplots()
for a in alphas:
    with np.errstate(all="ignore"):
        _, hist = gradient_descent(Xb_train, y_train, alpha=a, n_iters=200)
    if np.isfinite(hist[-1]) and hist[-1] < 10 * hist[0]:
        ax.plot(hist, label=f"α = {a}")
    else:
        print(f"α = {a}: diverged")
ax.set_xlabel("Iteration"); ax.set_ylabel("J(θ)"); ax.set_yscale("log")
ax.set_title("Effect of learning rate"); ax.legend(); plt.show()

In [ ]:
ALPHA, N_ITERS = 0.1, 5000
theta_gd, cost_hist = gradient_descent(Xb_train, y_train, alpha=ALPHA, n_iters=N_ITERS, tol=1e-12)

plt.plot(cost_hist)
plt.xlabel("Iteration"); plt.ylabel("J(θ)")
plt.title(f"Gradient descent convergence (α={ALPHA})")
plt.grid(alpha=0.3); plt.show()

print(f"Initial cost: {cost_hist[0]:.5f}")
print(f"Final cost  : {cost_hist[-1]:.5f}")
print("Monotonically decreasing:", bool(np.all(np.diff(cost_hist) <= 1e-12)))

## 7. Test-set evaluation (computed manually)

In [ ]:
y_pred_train = predict(Xb_train, theta_gd)
y_pred_test  = predict(Xb_test,  theta_gd)

results = pd.DataFrame({
    "MSE": [mse(y_train, y_pred_train), mse(y_test, y_pred_test)],
    "R²":  [r2_score_manual(y_train, y_pred_train), r2_score_manual(y_test, y_pred_test)],
}, index=["train", "test"])
results

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(y_test, y_pred_test, s=4, alpha=0.3)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax[0].plot(lims, lims, "r--"); ax[0].set_xlabel("Actual"); ax[0].set_ylabel("Predicted")
ax[0].set_title("Predicted vs actual (test)")
ax[1].hist(y_test - y_pred_test, bins=60); ax[1].set_title("Residuals (test)")
plt.tight_layout(); plt.show()

## 8. Sanity check against scikit-learn
Both models are fit on the same standardized training data, so coefficients are directly comparable.

In [ ]:
sk = LinearRegression().fit(X_train, y_train)
sk_pred_test = sk.predict(X_test)

# Closed-form normal equation (a third reference): theta = (X^T X)^-1 X^T y
theta_ne = np.linalg.solve(Xb_train.T @ Xb_train, Xb_train.T @ y_train)

coef_table = pd.DataFrame({
    "from-scratch GD": theta_gd,
    "sklearn":         np.r_[sk.intercept_, sk.coef_],
    "normal equation": theta_ne,
}, index=["intercept"] + feature_names)
coef_table["|GD − sklearn|"] = (coef_table["from-scratch GD"] - coef_table["sklearn"]).abs()
display(coef_table.round(6))

metrics = pd.DataFrame({
    "Test MSE": [mse(y_test, y_pred_test), mse(y_test, sk_pred_test)],
    "Test R²":  [r2_score_manual(y_test, y_pred_test), r2_score_manual(y_test, sk_pred_test)],
}, index=["from-scratch GD", "sklearn LinearRegression"])
display(metrics.round(6))

ok = np.allclose(theta_gd, np.r_[sk.intercept_, sk.coef_], atol=1e-3)
print("From-scratch coefficients match scikit-learn ✅" if ok else
      "Coefficients differ slightly: increase n_iters or alpha (see section 6)")
print("Max |prediction difference| on test set:", np.max(np.abs(y_pred_test - sk_pred_test)))

In [ ]:
# Markdown snippet to paste straight into your README
print(metrics.round(4).to_markdown())
print()
print(coef_table.round(4).to_markdown())

## 9. The math, in plain words (copy into your README)

**Model.** Prediction is a weighted sum of the features plus an intercept. Stacking a column of ones onto $X$ lets us write this as one matrix product, $\hat y = X\theta$.

**Cost.** Mean squared error measures how wrong we are on average. Squaring makes errors positive, punishes big misses more, and makes the cost a smooth bowl (convex), so there is a single global minimum. The extra ½ is there only so the derivative is clean:
$$J(\theta)=\frac1{2m}\sum_i(\hat y_i-y_i)^2.$$

**Gradient.** Differentiate with respect to one parameter $\theta_j$ using the chain rule:
$$\frac{\partial J}{\partial\theta_j}=\frac1{2m}\sum_i 2(\hat y_i-y_i)\cdot\frac{\partial \hat y_i}{\partial\theta_j}=\frac1m\sum_i(\hat y_i-y_i)\,x_{ij}.$$
The 2 cancels the ½. Doing all $j$ at once gives $\nabla J=\frac1m X^\top(X\theta-y)$.

**Gradient descent.** The gradient points uphill, so we step the opposite way: $\theta\leftarrow\theta-\alpha\nabla J$. The learning rate $\alpha$ sets the step size. Too small is slow; too large overshoots and diverges. Because $J$ is convex, repeated steps converge to the same $\theta$ as the closed-form normal equation and as scikit-learn.

**Why standardize?** If one feature ranges 0–10 and another 0–10,000, the cost surface is a long narrow valley and one learning rate cannot suit both directions. Standardizing makes the bowl round, so gradient descent converges fast.

**R².** $1-\text{SS}_{res}/\text{SS}_{tot}$: the fraction of target variance explained, compared with always predicting the mean (R²=0).